1) **Set up**
- Selenium installation
- Import required libraries

In [8]:
!pip install selenium


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [25]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
import time
from bs4 import BeautifulSoup
import requests
import re
import pandas as pd
from urllib.parse import urlparse

2. Definition of useful functions:
   - to do **automated scroll down** of the web page
   - to **normalize subcategory's name**

In [45]:
def scroll_down(driver):
    last_height = driver.execute_script("return document.body.scrollHeight")

    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

        # Wait to load page
        time.sleep(3)

        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

In [ ]:
def extract_subcat(link):
    path = urlparse(link).path  
    parts = path.strip('/').split('/')  
    if parts:
        return parts[-1]  
    return None

3. Initializing the **WebDriver** and sending it to the desired page

In [27]:
driver = driver = webdriver.Chrome()
Unieuro_home = 'https://www.unieuro.it/online/esplora'
driver.get(Unieuro_home)

4) Collecting the **URLs** that lead to the pages of the subcategories

In [28]:
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

In [30]:
subcategories_tags = soup.find_all('div', class_ = 'navigation__secondlevel-panel')

subcategories=[]
subcategories_links = []

for i in subcategories_tags:
    link = i.find('a').get('href')
    subcategories_links.append(link)
    subcategory = i.find('span').get_text()
    subcategories.append(subcategory)
    
subcategories_links = ['https://www.unieuro.it' + item for item in subcategories_links if 'www.unieuro' not in item]
subcategories_links = list(set(subcategories_links))

subcategories = [extract_subcat(i) for i in subcategories_links]
subcategories = [i.replace('-', ' ') for i in subcategories]

5) Function that collects **products** from the pages

In [31]:
def find_products(soup):
    prods = soup.find_all('a', class_ = 'gtmListing js--product-tile__title')
    
    products = []
    
    for tag in prods:
        product=tag.get_text(strip=True)
        products.append(product)
    
    return products

6. Function that collects the **prices** of products

In [46]:
def find_prices(soup):
    all_prices = soup.find_all('a', class_='prices__price product-tile__price gtmListing')
    prices = []

    for tag in all_prices:
        prior_price_attr = tag.get('data-tagmanager-prior-price', '')

        if prior_price_attr and 'Prezzo precedente' in prior_price_attr:
            prior_price_cleaned = prior_price_attr.replace('Prezzo precedente', '').strip()
            prices.append(prior_price_cleaned)
        else:
            integer = tag.find('span', class_='price__integer')
            decimal = tag.find('span', class_='price__decimal')

            if integer:
                price = integer.get_text()
                if decimal:
                    price += decimal.get_text()
                prices.append(price)

    return prices

7. Iterative collection of each **product** and **price** (in lists) from each page of each **subcategory**
   
NON RUNNARE I DUE CHUNK SOTTOSTANTI

In [32]:
all_products = []
all_prices = []
categories_column = [] 

In [33]:
for i in range(len(subcategories_links)):
    link = subcategories_links[i]
    subcategory = subcategories[i]
    driver.get(link)
    scroll_down(driver)
    scroll_down(driver)

    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    cat_products = find_products(soup)
    cat_prices = find_prices(soup)
    cat = [subcategory]*len(cat_products)

    print(subcategory, len(cat_products), len(cat_prices), len(cat))
    
    for i in cat_products:
        all_products.append(i)
        
    for i in cat:
        categories_column.append(i)
        
    for i in cat_prices:
        all_prices.append(i)

Mobilit 281 281 281
Accessori telefonia 1000 1000 1000
Audio Portatile 134 134 134
Accessori TV DVD 368 368 368
Cucina   Preparazione e cottura cibi 1000 1000 1000
Videocamere 47 47 47
Tempo Libero 711 711 711
Playstation 4 448 448 448
Playstation 3 30 30 30
Diffusori Audio Portatili 279 279 279
Trattamento Aria 351 351 351
Cordless e Telefonia Fissa 31 31 31
Obiettivi 246 246 246
Libri Musica Film 53 53 53
Cura della persona Salute e Benessere 961 961 961
Energia rinnovabile 62 62 62
Elettrodomestici da incasso 1000 1000 1000
Nintendo Switch 407 407 407
Car Audio e Video 27 27 27
DJ 15 15 15
Nintendo 3DS 39 39 39
Componenti per PC 62 62 62
Xbox One 167 167 167
Reti e Connettivit 459 459 459
Computer 48 48 48
Nintendo Switch 2 17 17 17
Tablet e eBook Reader 584 584 584
Casalinghi e articoli da regalo 1000 1000 1000
Hard Disk e Storage 134 134 134
Ottica 1 1 1
Illuminazione e materiale elettrico 666 666 666
Xbox 360 14 14 14
Valigie 68 68 68
Caff 500 500 500
Wearable 759 759 759
DVD e B

8. Saving the lists as columns of a **new dataset**

In [42]:
UnieuroDataset = pd.DataFrame()
UnieuroDataset['Product'] = all_products
UnieuroDataset['Unit Price'] = all_prices
UnieuroDataset['Category'] = categories_column

In [43]:
UnieuroDataset.to_csv('/Users/camillagentili/Downloads/DataManagement/UnieuroScarped.csv')